In [1]:
import json
from collections import Counter
import pandas as pd

FILES = {
    "tv_audio":          "data/target_pool_tv_&_audio.json",
    "small_appliances":  "data/target_pool_small_appliances.json",
    "large_appliances":  "data/target_pool_large_appliances.json",
}

pools = {name: json.load(open(path, encoding="utf-8")) for name, path in FILES.items()}
for name, data in pools.items():
    print(f"{name}: {len(data)} products")

tv_audio: 561 products
small_appliances: 1683 products
large_appliances: 3543 products


## Top-level keys (outside `specifications`)

In [3]:
for name, data in pools.items():
    n = len(data)
    counter = Counter()
    for p in data:
        for k, v in p.items():
            if k == "specifications":
                continue
            if v is not None and v != "" and v != [] and v != {}:
                counter[k] += 1
    df = pd.DataFrame(counter.most_common(), columns=["key", "count"])
    df["fill_%"] = (df["count"] / n * 100).round(1)
    print(f"\n=== {name} ({n} products) — top-level keys ===")
    print(df.to_string(index=False))


=== tv_audio (561 products) — top-level keys ===
      key  count  fill_%
      url    561   100.0
     name    561   100.0
 category    561   100.0
 retailer    561   100.0
reference    561   100.0
image_url    559    99.6
price_eur    184    32.8

=== small_appliances (1683 products) — top-level keys ===
      key  count  fill_%
      url   1683   100.0
     name   1683   100.0
 category   1683   100.0
 retailer   1683   100.0
reference   1683   100.0
image_url   1674    99.5
price_eur    827    49.1

=== large_appliances (3543 products) — top-level keys ===
      key  count  fill_%
      url   3543   100.0
     name   3543   100.0
 category   3543   100.0
 retailer   3543   100.0
reference   3543   100.0
image_url   3522    99.4
price_eur   2599    73.4


## Spec keys (inside `specifications`)

In [4]:
for name, data in pools.items():
    n = len(data)
    counter = Counter()
    for p in data:
        specs = p.get("specifications") or {}
        for k, v in specs.items():
            if v is not None and v != "":
                counter[k] += 1
    df = pd.DataFrame(counter.most_common(60), columns=["spec_key", "count"])
    df["fill_%"] = (df["count"] / n * 100).round(1)
    print(f"\n=== {name} ({n} products) — top 60 spec keys ===")
    print(df.to_string(index=False))


=== tv_audio (561 products) — top 60 spec keys ===
                                        spec_key  count  fill_%
                                           Marke    289    51.5
                                      Modellname    257    45.8
                       Konnektivitätstechnologie    247    44.0
                                      Hersteller    228    40.6
                          Enthaltene Komponenten    215    38.3
                                    Modellnummer    197    35.1
                                            ASIN    187    33.3
                              Produktabmessungen    186    33.2
                                           Farbe    184    32.8
                                      Formfaktor    166    29.6
                            Platzierung des Ohrs    164    29.2
                                 Rauschkontrolle    158    28.2
                                 Kopfhörerbuchse    150    26.7
                                        Impedanz    

## Sample values for high-signal spec keys

In [5]:
SIGNAL_KEYS = [
    "EAN", "GTIN", "EAN-Code",
    "Hersteller Modellnummer", "Hersteller Artikelnummer", "Herstellernummer",
    "Modellnummer", "Modellname", "Artikelnummer",
    "Marke", "Hersteller",
    "Bildschirmdiagonale (cm/Zoll)", "Bildschirmdiagonale in cm, Zoll",
    "Groesse", "Größe",
]

for name, data in pools.items():
    print(f"\n=== {name} — signal key fill rates ===")
    n = len(data)
    rows = []
    for key in SIGNAL_KEYS:
        count = sum(1 for p in data if (p.get("specifications") or {}).get(key) not in (None, ""))
        samples = list({str((p.get("specifications") or {}).get(key)) for p in data
                        if (p.get("specifications") or {}).get(key) not in (None, "")})[:5]
        rows.append({"key": key, "count": count, "fill_%": round(count/n*100, 1), "samples": samples})
    df = pd.DataFrame(rows)
    df = df[df["count"] > 0]
    print(df.to_string(index=False))


=== tv_audio — signal key fill rates ===
                          key  count  fill_%                                                                                         samples
                         GTIN     86    15.3                     [8718863045787, 4548736153448, 4049011209237, 6942351418032, 4049011197947]
      Hersteller Modellnummer     63    11.2                                   [400578, QE65Q7FAAUXXN, SRT40FG6733C, 65NANO80A6B.AEU, 55C6K]
                 Modellnummer    197    35.1                        [XH24AN550MV, JBLT770NCBLK, WHCH520L.CE7, WHCH520W.CE7, JBLSENSELITEBLK]
                   Modellname    257    45.8 [XH24AN550MV, JBL Endurance Run 3, GX-J92-Schwarz Bluetooth Kopfhörer, WHCH520W.CE7, WH-CH720N]
                        Marke    289    51.5                                                           [Kiano, CHIQ, SGRLRT, HitTid, UGREEN]
                   Hersteller    228    40.6           [Kiano, hama GmbH & Co KG, SGRLRT, HitTid, Vestel Holland

## Products missing key identifiers (no EAN, no model number)

In [6]:
MODEL_KEYS = ("Hersteller Modellnummer", "Hersteller Artikelnummer", "Herstellernummer", "Modellnummer", "Artikelnummer")
EAN_KEYS   = ("EAN", "GTIN", "EAN-Code")

for name, data in pools.items():
    n = len(data)
    no_ean   = [p for p in data if not p.get("ean") and not any((p.get("specifications") or {}).get(k) for k in EAN_KEYS)]
    no_model = [p for p in data if not any((p.get("specifications") or {}).get(k) for k in MODEL_KEYS)]
    no_both  = [p for p in no_ean if p in no_model]
    print(f"{name}: no EAN={len(no_ean)}/{n} ({len(no_ean)/n*100:.0f}%)  "
          f"no model#={len(no_model)}/{n} ({len(no_model)/n*100:.0f}%)  "
          f"missing both={len(no_both)}/{n} ({len(no_both)/n*100:.0f}%)")

tv_audio: no EAN=475/561 (85%)  no model#=301/561 (54%)  missing both=278/561 (50%)
small_appliances: no EAN=1258/1683 (75%)  no model#=879/1683 (52%)  missing both=762/1683 (45%)
large_appliances: no EAN=2857/3543 (81%)  no model#=1265/3543 (36%)  missing both=1153/3543 (33%)
